# Count CSC faults & script failures for a given night

MVP scope:
- Night window = sun altitude ≤ −12° at Cerro Pachón (nautical twilight bounds).
- CSC groups: MainTel, Camera, ESS (toggleable).
- Script failures from `Script.logevent_state`.
- Dedup: each script failure within `[t − pre, t + post]` of a CSC fault is attributed to the fault.
- Counts recovery cycles (FAULT → … → FAULT) as separate transitions.
- Dome subsystem faults: hook stubbed; see cell below.

In [2]:
import asyncio
import numpy as np
import pandas as pd
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u
from lsst_efd_client import EfdClient

## Configuration

In [13]:
EFD_NAME = "usdf_efd"          # or "summit_efd"
OBSERVING_DATE = "2026-05-16"  # local evening date (YYYY-MM-DD); night spans into the following UTC day

# Correlation window around each script failure (seconds)
CORRELATION_PRE_S  = 35.0   # generous to catch dome-style delayed failures
CORRELATION_POST_S = 5.0    # small slack for timestamp jitter

# Toggle groups
USE_GROUPS = {
    "maintel":         True,
    "camera":          True,
    "ess":             True,
    "dome_subsystems": False,  # flip when the hook below is implemented
}

## CSC groups

Edit freely. Format: `(csc_name, sal_index_or_None)`.

In [14]:
CSC_GROUPS = {
    "maintel": [
        ("MTMount",          None),
        ("MTPtg",            None),
        ("MTAOS",            None),
        ("MTRotator",        None),
        ("MTHexapod",        1),     # camera hexapod
        ("MTHexapod",        2),     # M2 hexapod
        ("MTM1M3",           None),
        ("MTM1M3TS",         None),
        ("MTM2",             None),
        ("MTDome",           None),
        ("MTDomeTrajectory", None),
    ],
    "camera": [
        ("MTCamera",         None),  # LSSTCam — confirm CSC name in your environment
        ("MTHeaderService",  None),
        ("MTOODS",           None),
    ],
    # Fill ESS indices that are actually deployed on the mountain for this night.
    "ess": [("ESS", idx) for idx in [1, 101, 102, 103, 104, 105, 106, 107, 108,
                                       201, 202, 203, 204, 205, 301]],
}

## Night window (sun ≤ −12° at Cerro Pachón)

In [15]:
RUBIN_LOC = EarthLocation.from_geodetic(
    lon=-70.749417*u.deg, lat=-30.244639*u.deg, height=2663*u.m,
)

def nautical_night(date_str, alt_limit_deg=-12.0):
    """Return (t_start, t_end) as astropy Times for sun altitude <= alt_limit_deg."""
    # Search noon-UTC to noon-UTC the next day on a 30-s grid (catches the full Chilean night).
    t0 = Time(f"{date_str}T12:00:00", scale="utc")
    times = t0 + np.linspace(0, 24, 2881) * u.hour
    alt = get_sun(times).transform_to(AltAz(obstime=times, location=RUBIN_LOC)).alt.deg
    below = alt <= alt_limit_deg
    if not below.any():
        raise ValueError(f"Sun never reaches {alt_limit_deg}° on {date_str}.")
    idx = np.where(below)[0]
    return times[idx[0]], times[idx[-1]]

t_start, t_end = nautical_night(OBSERVING_DATE)
print(f"Night window: {t_start.iso} → {t_end.iso}  (Δ = {(t_end - t_start).to(u.hour):.2f})")

Night window: 2026-05-16 22:51:30.000 → 2026-05-17 10:27:30.000  (Δ = 11.60 h)


## EFD client

In [16]:
client = EfdClient(EFD_NAME)

## CSC faults (transitions into `summaryState == FAULT`)

In [17]:
FAULT_STATE = 3  # SAL State enum: STANDBY=5, DISABLED=1, ENABLED=2, FAULT=3, OFFLINE=4

async def csc_fault_transitions(client, csc, idx, t0, t1):
    topic = f"lsst.sal.{csc}.logevent_summaryState"
    try:
        kw = {"index": idx} if idx is not None else {}
        df = await client.select_time_series(topic, ["summaryState"], t0, t1, **kw)
    except Exception as e:
        print(f"  [skip] {csc}:{idx}  ({type(e).__name__}: {e})")
        return pd.DataFrame()
    if df.empty:
        return df
    df = df.sort_index()
    # Drop consecutive duplicates so each FAULT row is a true transition into FAULT.
    df = df[df["summaryState"] != df["summaryState"].shift()]
    df = df[df["summaryState"] == FAULT_STATE].copy()
    df["csc"] = f"{csc}:{idx}" if idx is not None else csc
    return df

async def collect_csc_faults():
    selected = []
    for grp, on in USE_GROUPS.items():
        if grp == "dome_subsystems" or not on:
            continue
        selected.extend(CSC_GROUPS.get(grp, []))
    parts = []
    for name, idx in selected:
        df = await csc_fault_transitions(client, name, idx, t_start, t_end)
        if not df.empty:
            parts.append(df)
    return pd.concat(parts).sort_index() if parts else pd.DataFrame(columns=["summaryState", "csc"])

csc_faults = await collect_csc_faults()
print(f"CSC fault transitions: {len(csc_faults)}")
csc_faults["csc"].value_counts() if not csc_faults.empty else None

CSC fault transitions: 8


csc
MTPtg      6
MTMount    2
Name: count, dtype: int64

## Script failures (`Script.logevent_state` terminal failure states)

In [18]:
# ScriptState enum: FAILED=10, CONFIGURE_FAILED=11
SCRIPT_FAIL_STATES = (10, 11)

async def get_script_failures(client, t0, t1):
    df = await client.select_time_series(
        "lsst.sal.Script.logevent_state",
        ["state", "ScriptID"],
        t0, t1,
    )
    if df.empty:
        return df
    df = df.sort_index()
    df = df[df["state"].isin(SCRIPT_FAIL_STATES)]
    # One row per failed script execution.
    return df.drop_duplicates(subset="ScriptID", keep="first")

script_failures = await get_script_failures(client, t_start, t_end)
print(f"Script failures: {len(script_failures)}")
script_failures.head()

Script failures: 1


,state,ScriptID
2026-05-16 22:59:34.741790+00:00,10,None


## Dome subsystem faults (stub)

MTDome subsystems (AMCS, LWSCS, ApSCS, ThCS, MonCS, RAD) can enter degraded/fault states while `MTDome.summaryState` stays ENABLED. Three candidate signals to evaluate before wiring this in:

1. **Per-subsystem motion-state events** (`logevent_azMotion`, `logevent_elMotion`, `logevent_apertureMotion`, …) — check the state enum for a fault/error code.
2. **`logevent_logMessage` with `level >= 40`** (ERROR) — coarse but easy; many subsystem faults emit ERROR log lines.
3. **`logevent_errorCode`** with non-zero `errorCode` — cleanest if the CSC publishes it for the cases we care about.

Worth a short investigation in the EFD schema browser before committing to one. For now this returns empty.

In [19]:
async def get_dome_subsystem_faults(client, t0, t1):
    # TODO: implement once the signal of choice is confirmed.
    return pd.DataFrame()

dome_subs = await get_dome_subsystem_faults(client, t_start, t_end) if USE_GROUPS["dome_subsystems"] else pd.DataFrame()
print(f"Dome subsystem faults: {len(dome_subs)}")

Dome subsystem faults: 0


## Correlate & count

In [20]:
def attribute_scripts_to_faults(faults_df, scripts_df, pre_s, post_s):
    if scripts_df.empty:
        return scripts_df.assign(attributed=pd.Series(dtype=bool))
    if faults_df.empty:
        return scripts_df.assign(attributed=False)
    f_t = faults_df.index.values.astype("datetime64[ns]")
    pre  = np.timedelta64(int(pre_s  * 1e9), "ns")
    post = np.timedelta64(int(post_s * 1e9), "ns")
    flags = [((f_t >= t - pre) & (f_t <= t + post)).any()
             for t in scripts_df.index.values.astype("datetime64[ns]")]
    return scripts_df.assign(attributed=flags)

all_faults = pd.concat([csc_faults, dome_subs]) if not dome_subs.empty else csc_faults
scripts_marked = attribute_scripts_to_faults(all_faults, script_failures,
                                              CORRELATION_PRE_S, CORRELATION_POST_S)

n_csc_faults    = len(csc_faults)
n_dome_subs     = len(dome_subs)
n_scripts       = len(scripts_marked)
n_attributed    = int(scripts_marked["attributed"].sum()) if n_scripts else 0
n_standalone    = n_scripts - n_attributed
n_unique_total  = n_csc_faults + n_dome_subs + n_standalone

## Summary

In [21]:
print(f"""
═══════════════════════════════════════════════════════════
 Night: {OBSERVING_DATE}
 Window: {t_start.iso} → {t_end.iso} UTC  ({(t_end - t_start).to(u.hour):.2f})
 Correlation: script failure within [-{CORRELATION_PRE_S:.0f}s, +{CORRELATION_POST_S:.0f}s] of CSC fault
───────────────────────────────────────────────────────────
 CSC fault transitions:           {n_csc_faults}
 Dome subsystem faults:           {n_dome_subs}
 Script failures (total):         {n_scripts}
   ├─ attributed to CSC fault:    {n_attributed}
   └─ standalone:                 {n_standalone}
───────────────────────────────────────────────────────────
 TOTAL UNIQUE FAILURE EVENTS:     {n_unique_total}
═══════════════════════════════════════════════════════════
""")

if not csc_faults.empty:
    print("Faults by CSC:")
    print(csc_faults["csc"].value_counts().to_string())


═══════════════════════════════════════════════════════════
 Night: 2026-05-16
 Window: 2026-05-16 22:51:30.000 → 2026-05-17 10:27:30.000 UTC  (11.60 h)
 Correlation: script failure within [-35s, +5s] of CSC fault
───────────────────────────────────────────────────────────
 CSC fault transitions:           8
 Dome subsystem faults:           0
 Script failures (total):         1
   ├─ attributed to CSC fault:    1
   └─ standalone:                 0
───────────────────────────────────────────────────────────
 TOTAL UNIQUE FAILURE EVENTS:     8
═══════════════════════════════════════════════════════════

Faults by CSC:
csc
MTPtg      6
MTMount    2
